Imports:

In [1]:
from transformers import AutoTokenizer, AutoModel
import torch

Important values:

In [2]:
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
#TODO change this to large when accessing GPUs
tokenizer = AutoTokenizer.from_pretrained('bert-large-cased')
model = AutoModel.from_pretrained("SpanBERT/spanbert-large-cased")
max_size = 512
cls_token_id = 101
sep_token_id = 102

Some weights of BertModel were not initialized from the model checkpoint at SpanBERT/spanbert-base-cased and are newly initialized: ['bert.pooler.dense.weight', 'bert.pooler.dense.bias']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Return a list of tensors whose size is 512 tokens:

In [3]:
def spanbert_encode(text):
    #tokenize the text without special tokens, we will add them ourselves.
    tokenized_text = tokenizer.encode_plus(text, add_special_tokens=False, return_tensors='pt')

    #split the text in lists of less than 512 sub-words tokens (required for most BERT-like transformers), that overlap each other. Here 510 to add 2 special tokens.
    window_size = max_size - 2
    stride = int((max_size - 2) / 2)
    size = tokenized_text['input_ids'][0].size(0)

    print("Number of tokens in input:" + str(len(tokenized_text['input_ids'][0])))

    split_ids = [tokenized_text['input_ids'][0][i:i + window_size] for i in range(0, size, stride)]
    split_masks = [tokenized_text['attention_mask'][0][i:i + window_size] for i in range(0, size, stride)]

    #remove last chunk if too small
    last_chunk_size = len(split_ids[len(split_ids) - 1])
    if len(split_ids) > 1 and last_chunk_size < stride:
        split_ids.pop()
        split_masks.pop()
        last_chunk_size = len(split_ids[len(split_ids) - 1])

    #prepare input for SpanBERT
    for i in range(len(split_ids)):
        #add special tokens CLS and SEP
        split_ids[i] = torch.cat([torch.tensor([cls_token_id]), split_ids[i], torch.tensor([sep_token_id])])

        #add masks for the 2 special tokens
        split_masks[i] = torch.cat([torch.tensor([1]), split_masks[i], torch.tensor([1])])

        #we need to pad if the part is not 512 tokens long (only for last part)
        padding = max_size - len(split_ids[i])
        if padding > 0:
            split_ids[i] = torch.cat([split_ids[i], torch.Tensor([0] * padding)])
            split_masks[i] = torch.cat([split_masks[i], torch.Tensor([0] * padding)])

    #create dictionary for SpanBERT input.
    ids = torch.stack(split_ids).to(device)
    masks = torch.stack(split_masks).to(device)
    data_dict = {'input_ids': ids.long(), 'attention_mask': masks.int()}

    #get SpanBERT encoding
    model_device = model.to(device)
    spanbert_output = model_device(**data_dict).last_hidden_state  #float32 tensor
    spanbert_output_size = spanbert_output.size(dim=0)

    #create new tensor from overlapping one
    non_overlapping_tensors = []

    if spanbert_output_size > 1:
        #add first tensor
        non_overlapping_tensors.append(torch.narrow(spanbert_output[0], 0, 1, stride + stride // 2))
        #add middle tensors
        for i in range(1, spanbert_output_size - 1):
            non_overlapping_tensors.append(torch.narrow(spanbert_output[i], 0, stride // 2 + 1, stride))
        #add last tensor
        non_overlapping_tensors.append(torch.narrow(spanbert_output[spanbert_output_size - 1], 0, stride // 2 + 1, last_chunk_size - stride // 2))
    else:
        non_overlapping_tensors.append(torch.narrow(spanbert_output[0], 0, 1, last_chunk_size))

    output_tensor = torch.cat(non_overlapping_tensors, dim=0)

    print("Size of the encoded text tensor: " + str(output_tensor.size()))

    return output_tensor


Testing:

In [4]:
text1 = "London. Michaelmas term lately over, and the Lord Chancellor sitting in Lincoln's Inn Hall. Implacable November weather. As much mud in the streets as if the waters had but newly retired from the face of the earth, and it would not be wonderful to meet a Megalosaurus,forty feet long or so, waddling like an elephantine lizard up HolbornHill. Smoke lowering down from chimney-pots, making a soft black drizzle, with flakes of soot in it as big as full-grown snowflakes--gone into mourning, one might imagine, for the death ofthe sun. Dogs, undistinguishable in mire. Horses, scarcely better;splashed to their very blinkers. Foot passengers, jostling oneanother's umbrellas in a general infection of ill temper, and losing their foot-hold at street-corners, where tens of thousands of other foot passengers have been slipping and sliding since the day broke (if this day ever broke), adding new deposits to the crust upon crust of mud, sticking at those points tenaciously to the pavement, and accumulating at compound interest. Fog everywhere. Fog up the river, where it flows among green aits and meadows; fog down the river, where it rolls defiled among the tiers of shipping and the waterside pollutions of a great (and dirty) city. Fog on the Essex marshes, fog on the Kentish heights. Fog creeping into the cabooses of collier-brigs; fog lying out on the yards and hovering in the rigging of great ships; fog drooping on the gunwales of barges and small boats. Fog in the eyes and throats of ancient Greenwich pensioners, wheezing by the firesides of their wards; fog in the stem and bowl of the afternoon pipe of the wrathful skipper, down in his close cabin; fog cruelly pinching the toes and fingers of his shivering little 'prentice boy on deck. Chance people on the bridges peeping over the parapets into a nether sky of fog, with fog all round them, as if they were up in a balloon and hanging in the misty clouds. Gas looming through the fog in divers places in the streets, much at the sun may, from the spongey fields, be seen to loom by husbandman and ploughboy. Most of the shops lighted two hours before their time--as the gas seems to know, for it has a haggard and unwilling look. The raw afternoon is rawest, and the dense fog is densest, and the muddy streets are muddiest near that leaden-headed old obstruction, appropriate ornament for the threshold of a leaden-headed old corporation, Temple Bar. And hard by Temple Bar, in Lincoln's Inn Hall, at the very heart of the fog, sits the Lord High Chancellor in his High Court of Chancery. Never can there come fog too thick, never can there come mud and mire too deep, to assort with the groping and floundering condition which this High Court of Chancery, most pestilent of hoary sinners, holds this day in the sight of heaven and earth. London. Michaelmas term lately over, and the Lord Chancellor sitting in Lincoln's Inn Hall. Implacable November weather. As much mud in the streets as if the waters had but newly retired from the face of the earth, and it would not be wonderful to meet a Megalosaurus,forty feet long or so, waddling like an elephantine lizard up HolbornHill. Smoke lowering down from chimney-pots, making a soft black drizzle, with flakes of soot in it as big as full-grown snowflakes--gone into mourning, one might imagine, for the death ofthe sun. Dogs, undistinguishable in mire. Horses, scarcely better;splashed to their very blinkers. Foot passengers, jostling oneanother's umbrellas in a general infection of ill temper, and losing their foot-hold at street-corners, where tens of thousands of other foot passengers have been slipping and sliding since the day broke (if this day ever broke), adding new deposits to the crust upon crust of mud, sticking at those points tenaciously to the pavement, and accumulating at compound interest."

text2 = "I'm a penguin who likes icecream, I find it very good. My mother always said that it is bad for me to eat so much of it. I hope she is not be angry about it."

text3 = "London. Michaelmas term lately over, and the Lord Chancellor sitting in Lincoln's Inn Hall. Implacable November weather. As much mud in the streets as if the waters had but newly retired from the face of the earth, and it would not be wonderful to meet a Megalosaurus,forty feet long or so, waddling like an elephantine lizard up HolbornHill. Smoke lowering down from chimney-pots, making a soft black drizzle, with flakes of soot in it as big as full-grown snowflakes--gone into mourning, one might imagine, for the death ofthe sun. Dogs, undistinguishable in mire. Horses, scarcely better;splashed to their very blinkers. Foot passengers, jostling oneanother's umbrellas in a general infection of ill temper, and losing their foot-hold at street-corners, where tens of thousands of other foot passengers have been slipping and sliding since the day broke (if this day ever broke), adding new deposits to the crust upon crust of mud, sticking at those points tenaciously to the pavement, and accumulating at compound interest."

spanbert_encode(text1)

Token indices sequence length is longer than the specified maximum sequence length for this model (936 > 512). Running this sequence through the model will result in indexing errors


Number of tokens in input:936
Size of the encoded text tensor: torch.Size([936, 768])


tensor([[-1.4786e-01, -1.6402e-01, -1.4242e-01,  ...,  9.0508e-03,
          1.1350e-01,  1.3995e-01],
        [-1.8626e-01, -3.9554e-02, -1.9410e-02,  ..., -1.0139e-01,
          1.5508e-01, -9.8533e-03],
        [-4.0201e-02,  8.3199e-03, -1.2021e-01,  ..., -4.0480e-02,
          1.1707e-01,  1.0996e-01],
        ...,
        [ 1.3967e-01, -4.4615e-02, -2.1440e-01,  ..., -6.5149e-02,
          3.1588e-02,  4.7602e-05],
        [ 1.6899e-01,  3.0755e-03, -1.4416e-01,  ..., -5.9363e-02,
         -3.9273e-02, -1.5618e-02],
        [ 1.7393e-01, -4.9371e-02, -1.4419e-01,  ..., -1.4824e-01,
          1.1051e-01,  9.1543e-02]], device='cuda:0', grad_fn=<CatBackward0>)

Now we have the SpanBERT encoding, we can finalize the encoding with an attention mechanism: